# Search + URN Navigation

## Table of contents (ToC) <a class="anchor" id="TOC"></a>
* <a href="#introduction">1 - Introduction</a>
* <a href="#setup">2 - Setup</a>
* <a href="#search-scaife">3 - Search Scaife</a>
* <a href="#navigate-cts">4 - Navigate CTS references</a>
* <a href="#sources">5 - Sources</a>
* <a href="#required-libraries">6 - Required libraries</a>
* <a href="#notebook-version">7 - Notebook version</a>

## 1 - Introduction <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook demonstrates Scaife JSON search and CTS navigation from ordered valid references. The live Perseus CTS service may return malformed HTML for `GetPrevNextUrn`, so the navigation example derives neighbors from `GetValidReff`, matching the fallback strategy implemented by the MCP server.

The edition URN is an example recorded from the live CTS inventory. Discover the current edition before reusing it in other workflows.

## 2 - Setup <a class="anchor" id="setup"></a>
##### [Back to ToC](#TOC)

We use `%pip` instead of `!pip` because `%pip` installs into the current Jupyter kernel.

In [1]:
%pip install --quiet httpx

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import xml.etree.ElementTree as ET

import httpx

CTS_BASE = "https://www.perseus.tufts.edu/hopper/CTS"
SEARCH_BASE = "https://scaife.perseus.org/search/json/"

## 3 - Search Scaife <a class="anchor" id="search-scaife"></a>
##### [Back to ToC](#TOC)

Submit a Unicode Greek form query to Scaife's JSON library-search endpoint and inspect the first page of results.

In [3]:
query = "μῆνιν"
search = httpx.get(
    SEARCH_BASE,
    params={"q": query, "kind": "form", "type": "library", "page_num": 1},
    timeout=20.0,
)
search.raise_for_status()
print(json.dumps(search.json(), ensure_ascii=False, indent=2)[:1200])

{
  "results": [
    {
      "passage": {
        "url": "/reader/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238/",
        "json_url": "/library/passage/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238/json/",
        "text_url": "/library/passage/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2:45.238/text/",
        "text": {
          "url": "/library/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2/",
          "json_url": "/library/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2/json/",
          "text_url": "/library/passage/urn:cts:greekLit:tlg2045.tlg001.perseus-grc2/text/",
          "ancestors": [
            {
              "url": "/library/urn:cts:greekLit:tlg2045/",
              "json_url": "/library/urn:cts:greekLit:tlg2045/json/",
              "text_url": "/library/passage/urn:cts:greekLit:tlg2045/text/",
              "urn": "urn:cts:greekLit:tlg2045",
              "label": "Nonnus of Panopolis"
            },
            {
              "url": "/library/urn:cts:greekL

## 4 - Navigate CTS references <a class="anchor" id="navigate-cts"></a>
##### [Back to ToC](#TOC)

Retrieve ordered valid references from CTS and derive the previous and next citations around a selected passage.

In [4]:
edition = "urn:cts:greekLit:tlg0012.tlg001.perseus-grc1"
current = f"{edition}:1.10"
refs = httpx.get(
    CTS_BASE,
    params={"request": "GetValidReff", "urn": edition},
    timeout=20.0,
)
refs.raise_for_status()

root = ET.fromstring(refs.text)
urns = [element.text for element in root.iter() if element.tag.rsplit("}", 1)[-1] == "urn"]
index = urns.index(current)
print("previous:", urns[index - 1])
print("current: ", current)
print("next:    ", urns[index + 1])

previous: urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.9
current:  urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.10
next:     urn:cts:greekLit:tlg0012.tlg001.perseus-grc1:1.11


## 5 - Sources <a class="anchor" id="sources"></a>
##### [Back to ToC](#TOC)

This notebook uses live data from:

- the [Perseus Digital Library](https://www.perseus.tufts.edu/);
- the Perseus [CTS endpoint](https://www.perseus.tufts.edu/hopper/CTS), including `GetPassage` and `GetValidReff`;
- the local project guidance in the [README](../README.md).

The upstream CTS inventory and responses can change. Prefer discovering available resources at runtime instead of treating example edition URNs as permanent identifiers.

Search results are supplied by the [Scaife Viewer](https://scaife.perseus.org/) JSON search service.

## 6 - Required libraries <a class="anchor" id="required-libraries"></a>
##### [Back to ToC](#TOC)

This repository targets **Python 3.11 or newer**. The notebook directly requires:

- `httpx>=0.27.0` for HTTP requests;
- Jupyter/IPython to run the cells.

The notebook's `%pip install --quiet httpx` cell installs `httpx` into the active Jupyter kernel. Alternatively, install all project dependencies from the repository root with `pip install -e .`.

The standard-library modules `json` and `xml.etree.ElementTree` require no additional installation.

## 7 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr>
      <td><strong>Author</strong></td>
      <td>Tony Jurg</td>
    </tr>
    <tr>
      <td><strong>Version</strong></td>
      <td>1.1</td>
    </tr>
    <tr>
      <td><strong>Date</strong></td>
      <td>June 18, 2026</td>
    </tr>
  </table>
</div>